# K Nearest Neighbor with Learned Similarity
In DINOv3, they study the cosine simliarity of a patch with other patches. They observe the cosine similarity patch becomes more noisy as training iterates. We want to observe the knn metric's change throughout the training. 

The follwing image is from DINOv3:
![Description of the image](cosine_similarity.png)

In [77]:
import torch
import numpy as np
import torchvision
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision import transforms
from pathlib import Path
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

PyTorch: 2.9.1
Torchvision: 0.24.1


In [ ]:
base_dir = (
    Path.home()
    / "Documents"
    / "Toronto Graduate Study"
    / "MASc Thesis"
    / "World_Model"
    / "Neural_Language"
    / "KNN_attention"
)

REPO_DIR = base_dir / "dinov3-main"

vits16_weight_path = base_dir / "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
vits16plus_weight_path = base_dir / 'dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth'
vitb16_weight_path = base_dir / 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'

davis_img_sample_path = base_dir / 'dataset/DAVIS/JPEGImages/Full-Resolution/bear/00000.jpg'

In [56]:
# DINOv3 ViT models pretrained on web images
dinov3_vits16 = torch.hub.load(REPO_DIR, 'dinov3_vits16', source='local', weights=str(vits16_weight_path))
dinov3_vits16plus = torch.hub.load(REPO_DIR, 'dinov3_vits16plus', source='local', weights=str(vits16plus_weight_path))
dinov3_vitb16 = torch.hub.load(REPO_DIR, 'dinov3_vitb16', source='local', weights=str(vitb16_weight_path))


Downloading: "file:///Users/chengxinye/Documents/Toronto%20Graduate%20Study/MASc%20Thesis/World_Model/Neural_Language/KNN_attention/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth" to /Users/chengxinye/.cache/torch/hub/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 327M/327M [00:00<00:00, 1.49GB/s]


In [ ]:
def cosine_similarity(patch_a, patch_b):
    '''
    Compute cosine similarity between two patches or feature vectors.

    Input:
        PIL images, NumPy arrays, or PyTorch tensors.

    Returns:
        A Python float between -1 and 1.
    '''
    def to_vector(patch):
        if not isinstance(patch, torch.Tensor):
            patch = torch.as_tensor(np.array(patch))

        return patch.to(dtype=torch.float32).reshape(-1)

    a = to_vector(patch_a)
    b = to_vector(patch_b).to(a.device)

    if a.numel() != b.numel():
        raise ValueError("Inputs must contain the same number of elements")

    norm_a = torch.linalg.vector_norm(a)
    norm_b = torch.linalg.vector_norm(b)

    if norm_a.item() == 0 or norm_b.item() == 0:
        raise ValueError("Division by zero")

    score = torch.dot(a / norm_a, b / norm_b)

    return score.item()

In [ ]:
def image_patch_gram(image, patch_size = 16):
     """
    Compute a cosine Gram matrix using raw image patches.

    Incomplete patches along the bottom/right edges are excluded.
    """
     image = image.convert("RGB")
     width, height = image.size

     rows = height // patch_size
     cols = width // patch_size
     
     # Extract patches in row-major order
     patches = [
         image.crop((
             col * patch_size,
             row * patch_size,
             (col + 1) * patch_size,
             (row + 1) * patch_size,
         ))
         for row in range(rows)
         for col in range(cols)
     ]

     N = len(patches)
     G = np.empty((N,N))
     
     # Compute one triangle; cosine similarity is symmetric
     for i in range(N):
         for j in range(i, N):
             score = cosine_similarity(patches[i], patches[j])
             G[i, j] = score
             G[j, i] = score

     return G, (rows, cols)

In [92]:
model = dinov3_vits16
model.eval()

image = ImageOps.exif_transpose(
    Image.open(str(davis_img_sample_path))
).convert("RGB")

image = image.resize((512,512), Image.Resampling.BICUBIC)

# crop coordinates: (left, top, right, bottom)
patch_a = image.crop((0, 0, 16, 16))
patch_b = image.crop((32, 48, 48, 64))

score = cosine_similarity(patch_a, patch_b)
print("Pixel similarity:", score)

preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.485, 0.456, 0.406),
        std = (0.229, 0.224, 0.225)
    ),
])

parameter = next(model.parameters())

x = preprocess(image).unsqueeze(0).to(
    device = parameter.device,
    dtype = parameter.dtype
)

with torch.inference_mode():
    output = model.forward_features(x)
    tokens = output["x_norm_patchtokens"][0].float().cpu()

print(tokens.shape)

Pixel similarity: 0.9735691547393799
torch.Size([1024, 384])
